# Model Selection from Multiple Algorithm
options will be like
  - XGBoost (XGB)
  - LightGBM
  - Random Forest (RF)
  - Support Vector Machine (SVM)
  - K-Nearest Neighbors (KNN)
  - Logistic Regression
  - Naive Bayes

Not trying:
  - Gradient Boosting (Generally, XGB and LightGBM perform better than it)
  - Adabost (Generally, XGB and LightGBM perform better than it)
  - Decision Tree (Generally, RF perform better than it)
  - Catboost (all features are not categorical)

Also perform hyperparameter tunning on the above algorithms using Bayesian Optimization (e.g., Optuna) and log the best result of an algorithm as a run.

NOTE: here, Experiment 5 has been done in different files as Sequentially running of these algorithm takes a lot of time.

**Random Forest (RF)**

In [3]:
import os
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] = userdata.get('AWS_DEFAULT_REGION')
os.environ["satya_mlflow_ec2_uri"] = userdata.get('satya_mlflow_ec2_uri')

In [4]:
!pip install mlflow boto3 awscli optuna imbalanced-learn
!aws sts get-caller-identity

{
    "UserId": "AIDA3R56L7L2JBQ2C327X",
    "Account": "794431322868",
    "Arn": "arn:aws:iam::794431322868:user/satya-user-iam"
}


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import pandas as pd
import mlflow.sklearn
import mlflow, optuna


In [7]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Youtube_Comment_Sentiment_Analysis/reddit_preprocessing.csv').dropna()
df.shape

Mounted at /content/drive


(36662, 2)

In [6]:
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri(os.environ["satya_mlflow_ec2_uri"])

# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location='s3://satya-mlflow-bucket/188737626132630835', creation_time=1758703381598, experiment_id='188737626132630835', last_update_time=1758703381598, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [ ]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for Random Forest

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 10000  # Set max_features to 10000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, name=f"{model_name}_model")


# Step 6: Optuna objective function for Random Forest
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)  # Number of trees in the forest
    max_depth = trial.suggest_int('max_depth', 3, 20)  # Maximum depth of the tree
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)  # Minimum samples required to split a node
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)  # Minimum samples required at a leaf node

    # RandomForestClassifier setup
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                   min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
                                   random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for Random Forest, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_rf, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = RandomForestClassifier(n_estimators=best_params['n_estimators'],
                                        max_depth=best_params['max_depth'],
                                        min_samples_split=best_params['min_samples_split'],
                                        min_samples_leaf=best_params['min_samples_leaf'],
                                        random_state=42)

    # Log the best model with MLflow, passing the algo_name as "RandomForest"
    log_mlflow("RandomForest", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for Random Forest
run_optuna_experiment()


[I 2025-09-24 09:25:39,713] A new study created in memory with name: no-name-799075fa-dc3c-4277-992a-7318643e62e1
[I 2025-09-24 09:25:49,646] Trial 0 finished with value: 0.6909744240118368 and parameters: {'n_estimators': 261, 'max_depth': 16, 'min_samples_split': 10, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.6909744240118368.
[I 2025-09-24 09:25:52,807] Trial 1 finished with value: 0.6521876981610654 and parameters: {'n_estimators': 122, 'max_depth': 8, 'min_samples_split': 11, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.6909744240118368.
[I 2025-09-24 09:25:55,417] Trial 2 finished with value: 0.655992390615092 and parameters: {'n_estimators': 101, 'max_depth': 9, 'min_samples_split': 18, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.6909744240118368.
[I 2025-09-24 09:25:59,658] Trial 3 finished with value: 0.6932995138448531 and parameters: {'n_estimators': 124, 'max_depth': 20, 'min_samples_split': 20, 'min_samples_leaf': 18}. Best is trial 3 with va

🏃 View run RandomForest_SMOTE_TFIDF_Trigrams at: http://65.2.37.109:5000/#/experiments/188737626132630835/runs/0a0ad44bfa8841918d7d5ded28853c20
🧪 View experiment at: http://65.2.37.109:5000/#/experiments/188737626132630835
